----

# CMSE401 Exam Instructions

You will have the entire exam time (7:45am-9:45am) to complete the exam.   

Please read the following instructions before starting the exam.


> This is an open Internet exam.  Feel free to use anything on the Internet with one important exception...
> 
> - **DO NOT** communicate live with other people or AI tools during the exam (either verbally or online).  The goal here is to find answers to problems as you would in the real world and demonstrate your _own_ ability to solve problems.  
> 
> You will be given **until the end of the exam period** to complete this exam.  Use your time wisely. 
>
> There are a total of 145 points available on this exam. The exam will only be graded out of 125 points. While you can't receive a grade over 100%, this gives some flexiblity in how you approach this exam. For example, most questions are worth 10 points, so you could potentially skip 2 questions if there are two questions that are stumping you or would take too long to answer. 
>
> When grading, if you achieve 125 points or more, I will stop grading your exam and give you a grade of 100%. 
> 
> **HINTS:**
> - Neatness and grammar are important.  We will ignore all notes or code we can't read or understand.
> - Read the entire exam from beginning to end before starting.  Not all questions are equal in **points vs. time** so plan your time accordingly. 
> - Spaces for answers are provided. Delete the prompting text such as "Put your answer to the above question here" and replace it with your answer. Do not leave the prompting text with your answer.
> - Do not assume that the answer must be in the same format of the cell provided. Feel free to change the cell formatting (e.g., markdown to code, and vice versa) or add additional cells as needed to provide your answer.
> - When a question asks for an answer "**in your own words**" it is still okay to search the Internet for the answer as a reminder. *However*, we would like you to do more than cut and paste.  Make the answer your own. 
> - If you get stuck, try not to leave an answer blank. It is better to include some notes or stub functions so we have an idea about your thinking process so we can give you partial credit.   
> - Always provid links to any references you find helpful. 
> - Feel free to delete the provided check marks (&#9989;) as a way to keep track of which questions you have successfully completed. 
> - Many questions do not actually require you to run code. You may find it helpful to run the code to check your work, but you are not required to run the code unless specified. 

> **Honor Code**
> 
> I, agree to neither give nor receive any help on this exam from other people.  I also understand that providing answers to questions on this exam to other students is also an academic misconduct violation as is live communication or receiving answers to questions on this exam from other people. It is important to me to be a person of integrity and that means that ALL ANSWERS on this exam are my answers.
> 
> &#9989; **<font color=red>DO THIS:</font>** Include your name in the line below to acknowledge the above statement:

Put your name here.

___

# Section 1. (45 pts)
## High-Performance Computing (HPC)

This section will focus on conceptual questions related to HPC before we get into coding specifics in later sections. 

&#9989; **<font color=red>Question 1.1</font>**: (5 points) Why do we care about parallel computing? What is the key benefit over serial computing? 

***Your answer here***

&#9989; **<font color=red>Question 1.2</font>**: (10 points)  What is the difference between a weak scaling study and a strong scaling study? Provide an example of when each would be useful. 

***Your answer here***

&#9989; **<font color=red>Question 1.3</font>**: (10 points) In your own words, define shared-network parallelism? Which method(s) did we learn in this course that utilizes shared-network parallelism? 

***Your answer here***

&#9989; **<font color=red>Question 1.4</font>**: (10 points) Assume you are given two versions of some code, one that uses OpenMP and one that uses MPI. While both versions of the code are correct and produce the same output, the OpenMP code runs MUCH faster. In both scenarios, 10 cores are used. What could potentially be causing the MPI version to run slower or the OpenMP version to run faster? 

***Your answer here***

&#9989; **<font color=red>Question 1.5</font>**: (10 points) You are presented with a problem with the following properties: 

- The code requires you to run the same process on many different parts of a large data array 
- The primary bottleneck when running in serial is handling the sheer quantity of data
- The process doesn't require any intermediate communication/coordination, only one communication of the results at the end of applying the process to the different data regions

Which parallel method(s) would you suggest to apply to this problem? **Explain your reasoning!**

___

# Section 2. (30 pts)

## OpenMP

This section will focus on your ability to utilize OpenMP parallelism. You will use the following serial starter code for this section. This code implements the 2D heat equation on a 1000x1000 board over 100 timesteps. 

```c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define NX 1000
#define NY 1000
#define NSTEPS 100
#define ALPHA 1.0

double dx = 1.0 / (NX - 1);
double dy = 1.0 / (NY - 1);
double dt;

void save_frame(double u[NX][NY], int step) {
    char filename[64];
    snprintf(filename, sizeof(filename), "heat_step_%03d.dat", step);
    FILE *f = fopen(filename, "w");
    for (int i = 0; i < NX; i++) {
        for (int j = 0; j < NY; j++) {
            fprintf(f, "%f ", u[i][j]);
        }
        fprintf(f, "\n");
    }
    fclose(f);
}


int main() {
    double u[NX][NY], unew[NX][NY];
    dt = 0.25 * fmin(dx*dx, dy*dy) / ALPHA;


    //Initialize heatmap
    for (int i = 0; i < NX; i++) {
        for (int j = 0; j < NY; j++) {
            double x = i * dx;
            double y = j * dy;
            u[i][j] = exp(-100 * ((x - 0.5)*(x - 0.5) + (y - 0.5)*(y - 0.5)));
        }
    }

    //time loop running NSTEPS timesteps
    for (int n = 0; n < NSTEPS; n++) {
        //heatmap timestep
    	for (int i = 1; i < NX - 1; i++) {
        	for (int j = 1; j < NY - 1; j++) {
            	double dudx2 = (u[i+1][j] - 2*u[i][j] + u[i-1][j]) / (dx*dx);
            	double dudy2 = (u[i][j+1] - 2*u[i][j] + u[i][j-1]) / (dy*dy);
            	unew[i][j] = u[i][j] + ALPHA * dt * (dudx2 + dudy2);
        	}
    	}
        

        //boundary condition
        for (int i = 0; i < NX; i++) {
            u[i][0] = 0.0;
            u[i][NY - 1] = 0.0;
        }
        for (int j = 0; j < NY; j++) {
            u[0][j] = 0.0;
            u[NX - 1][j] = 0.0;
        }

        //replace heatmap
        for (int i = 0; i < NX; i++)
            for (int j = 0; j < NY; j++)
                u[i][j] = unew[i][j];
        
        //output heatmap if interested in verifying result (Note: This will slow down code substantially)
//        save_frame(u, n);
    }

    return 0;
}


```

&#9989; **<font color=red>Question 2.1</font>**: (5 points) Copy the above code into a c file on the HPCC. Below paste the code used to compile the code. 

***Your answer here***

&#9989; **<font color=red>Question 2.2</font>**: (5 points) Now using the compiled code. Benchmark the code. Include both the code used to run the benchmark and the benchmark result below. 

***Benchmarking Code: <here>***


***Benchmarking Results: <here>s***

&#9989; **<font color=red>Question 2.3</font>**: (10 points) Optimize the code using OpenMP. In the next question we will run it with 10 threads. (Note: You do not need to fully optimize the code. Just target a part of the code that you think would most benefit from OpenMP parallelism while keeping the result accurate.) Paste your updated code here, then explain your reasoning for the changes. 

```c


```

***Explain your changes and reasoning here***

&#9989; **<font color=red>Question 2.4</font>**: (10 points) Write the code below used to compile the updated code file. Once compiled, run a benchmark of the new code and record the time. Did you get a speedup?

***Code to compile: <here>***

***Runtime: <here>***

***Speedup?: <here>***

___

# Section 3 (30 pts)

## CUDA

This section will focus on your ability to utilize CUDA parallelism. The following code is a 1D version of the heat equation. 

```c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define NX 1000
#define NSTEPS 1000000
#define ALPHA 1.0

double dx = 1.0 / (NX - 1);
double dt;

void save_frame(double u[NX], int step) {
    char filename[64];
    snprintf(filename, sizeof(filename), "heat_step_%03d.dat", step);
    FILE *f = fopen(filename, "w");
    for (int i = 0; i < NX; i++) {
            fprintf(f, "%f ", u[i]);
    }
    fclose(f);
}


int main() {
    double u[NX], unew[NX];
    dt = 0.25 * dx*dx / ALPHA;


    //Initialize heatmap
    for (int i = 0; i < NX; i++) {
            double x = i * dx;
            u[i] = exp(-100 * ((x - 0.5)*(x - 0.5)));
            unew[i]=0.0;
    }

    //time loop running NSTEPS timesteps
    for (int n = 0; n < NSTEPS; n++) {
        //heatmap timestep
    	for (int i = 1; i < NX - 1; i++) {
            	double dudx2 = (u[i+1] - 2*u[i] + u[i-1]) / (dx*dx);
            	unew[i] = u[i] + ALPHA * dt * (dudx2);
    	}
        

        //replace heatmap
        for (int i = 0; i < NX; i++)
            u[i] = unew[i];
        
        //output heatmap if interested in verifying result (feel free to comment out to avoid file clutter)
        //if (n%100==0)
        //    save_frame(u, n);
    }
    //output final heatmap
    save_frame(u,NSTEPS);
    return 0;
}



```

&#9989; **<font color=red>Question 3.1</font>**: (10 points) Considering the above code, identify a region of the code that would be good to parallelize using a GPU and write the CUDA kernel. Paste the definition for the CUDA kernel below. 

```c

```

&#9989; **<font color=red>Question 3.2</font>**: (10 points) Below is a version of the main function adapted to work with CUDA, but there is a **problem** with this code. Identify the problem, explain why it is a problem, and explain how you would fix it. You can assume the kernel function exists and that it is correct. I left commented out code from the serial version so you can easily see what was replaced by the kernel function. (Hint: The problem is not an error, but it is still bad!)

```c
int main() {
    double u[NX], unew[NX];
    dt = 0.25 * dx*dx / ALPHA;
    double *u_c, *unew_c;
    cudaMalloc((void**)&u_c, NX*sizeof(double));
    cudaMalloc((void**)&unew_c, NX*sizeof(double));

    //Initialize heatmap
    for (int i = 0; i < NX; i++) {
            double x = i * dx;
            u[i] = exp(-100 * ((x - 0.5)*(x - 0.5)));
            unew[i]=0.0;
    }


    int threads=998;
    int blocks=1;

    //time loop running NSTEPS timesteps
    for (int n = 0; n < NSTEPS; n++) {
        //heatmap timestep
    	//for (int i = 1; i < NX - 1; i++) {
        //    	double dudx2 = (u[i+1] - 2*u[i] + u[i-1]) / (dx*dx);
        //    	unew[i] = u[i] + ALPHA * dt * (dudx2);
    	//}
        cudaMemcpy(u_c, u, NX*sizeof(double), cudaMemcpyHostToDevice);
        cudaMemcpy(unew_c, unew, NX*sizeof(double), cudaMemcpyHostToDevice); 
        step<<<blocks, threads>>>(u_c,unew_c, dx, dt);
        cudaMemcpy(u, u_c, NX*sizeof(double), cudaMemcpyDeviceToHost);
        cudaMemcpy(unew, unew_c, NX*sizeof(double), cudaMemcpyDeviceToHost);

        //replace heatmap
        //for (int i = 0; i < NX; i++)
        //    u[i] = unew[i];
        
        //output heatmap if interested in verifying result (feel free to comment out to avoid file clutter)
        //if (n%100==0)
        //    save_frame(u, n);
    }
    //output final heatmap
    save_frame(u, NSTEPS); 
    return 0;
}
```

***Explain the problem and your solution here***

&#9989; **<font color=red>Question 3.3</font>**: (10 points) Assume you have a working CUDA implementation of the heat equation saved in a file called `heat.cu`. What code would you need to compile and then run the code. If there are any modules you need to load, include those as well. 

```bash
#replace with your code
```

---

# Section 4 (40 pts)

## MPI

Below we have an adapted version of the 1d heat equation that is in a ring rather than a line. The key here is that the ends wrap around so heat can be transferred between the two ends. 

```c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define NX 10
#define NSTEPS 100
#define ALPHA 1.0

double dx = 1.0 / (NX - 1);
double dt;

void save_frame(double u[NX], int step) {
    char filename[64];
    snprintf(filename, sizeof(filename), "heat_step_%03d.dat", step);
    FILE *f = fopen(filename, "w");
    for (int i = 0; i < NX; i++) {
            fprintf(f, "%f ", u[i]);
    }
    fclose(f);
}


int main() {
    double u[NX], unew[NX];
    double dudx2;
    dt = 0.25 * dx*dx / ALPHA;


    //Initialize heatmap
    for (int i = 0; i < NX; i++) {
            double x = i * dx;
            u[i] = exp(-100 * ((x - 0.5)*(x - 0.5)));
            unew[i]=0.0;
    }

    //time loop running NSTEPS timesteps
    for (int n = 0; n < NSTEPS; n++) {
        //heatmap timestep
    	for (int i = 1; i < NX - 1; i++) {
            	dudx2 = (u[i+1] - 2*u[i] + u[i-1]) / (dx*dx);
            	unew[i] = u[i] + ALPHA * dt * (dudx2);
    	}
        
        // index 0 update
        dudx2 = (u[1] - 2*u[0] + u[NX-1])/ (dx*dx);
        unew[0] = u[0] + ALPHA * dt * (dudx2);

        // index NX-1 update
        dudx2 = (u[0] - 2*u[NX-1] + u[NX-2])/ (dx*dx);
        unew[NX-1] = u[NX-1] + ALPHA * dt * (dudx2); 

        //replace heatmap
        for (int i = 0; i < NX; i++)
            u[i] = unew[i];
        
        //output heatmap if interested in verifying result (feel free to comment out to avoid file clutter)
        if (n%1000==0)
            save_frame(u, n);
    }
    //output final heatmap
    save_frame(u,NSTEPS);
    return 0;
}


```

&#9989; **<font color=red>Question 4.1</font>**: (10 points) Below is some starter code that mostly adapts the above code to work with MPI. You will need to complete the message passing for the edge cases (rank==0 and rank==NX-1). Use the code below and add in your message passing functions where indicated. (Hint: I would recommend copying and pasting the supplied send receive calls and just modifying the necessary parts for the boundary cases.)

```c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <mpi.h>

#define NX 10
#define NSTEPS 1000
#define ALPHA 1.0

double dx = 1.0 / (NX - 1);
double dt;

void save_frame(double u[NX], int step) {
    char filename[64];
    snprintf(filename, sizeof(filename), "heat_step_%03d.dat", step);
    FILE *f = fopen(filename, "w");
    for (int i = 0; i < NX; i++) {
            fprintf(f, "%f ", u[i]);
    }
    fclose(f);
}


int main(int argc, char *argv[]) {
    double u[NX], unew[NX];
    double dudx2;
    int rank, size;
    MPI_Status status[4];
    MPI_Request reqs[4];
    dt = 0.25 * dx*dx / ALPHA;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);


    //Initialize heatmap
    for (int i = 0; i < NX; i++) {
            double x = i * dx;
            u[i] = exp(-100 * ((x - 0.5)*(x - 0.5)));
            unew[i]=0.0;
    }


    //set initial local values
    double left, center, right;
    double localnew;
    if (rank>0 && rank < NX-1){
        left = u[rank-1];
        center = u[rank];
        right = u[rank+1];
    }
    localnew=unew[rank];

    if (rank==0){
        left = u[NX-1];
        center = u[rank];
        right = u[rank+1];
    }
    if (rank==NX-1){
        left = u[rank-1];
        center = u[rank];
        right = u[0];
    }



    //time loop running NSTEPS timesteps
    for (int n = 0; n < NSTEPS; n++) {
        //heatmap timestep
        // message passing for all non-boundary ranks
        if (rank >0 && rank < NX-1){
            MPI_Isend(&center, 1, MPI_DOUBLE, rank+1, 0, MPI_COMM_WORLD, &reqs[0]);
            MPI_Isend(&center, 1, MPI_DOUBLE, rank-1, 0, MPI_COMM_WORLD, &reqs[1]);
            MPI_Irecv(&left, 1, MPI_DOUBLE, rank-1, 0, MPI_COMM_WORLD, &reqs[2]);
            MPI_Irecv(&right, 1, MPI_DOUBLE, rank+1, 0, MPI_COMM_WORLD, &reqs[3]); 
        }
        //message passing for rank 0
        if (rank==0){
            //Place send/receive calls here

        }
        //message passing for rank NX-1
        if (rank==NX-1){
            //Place send/receive calls here
            
        }

        //wait for all non-blocking message passing to finish
        MPI_Waitall(4, reqs, status);

        dudx2 = (right - 2*center + left) / (dx*dx);
        localnew = center + ALPHA * dt * (dudx2);
        
        //replace center value
        center = localnew;
    }
    
    //For Question 4.4
    MPI_Gather(&center, 1, MPI_DOUBLE, u, 1, MPI_DOUBLE, 0, MPI_COMM_WORLD);
    if (rank==0){
        //output final heatmap
        save_frame(u,NSTEPS);
    }
    MPI_Finalize();
    return 0;
}

```

&#9989; **<font color=red>Question 4.2</font>**: (10 points) What code would you use to compile and run the MPI code. Have the code you use to run the code specify that 10 processors will be used. (Note: You do not actually need to run the code, just write it here.)

```bash

```

&#9989; **<font color=red>Question 4.3</font>**: (10 points) Would you expect this MPI code to run faster or slower than the serial code? Explain why!


***Write your answer and reasoning here.***

&#9989; **<font color=red>Question 4.4</font>**: (10 points) In the supplied MPI code above, explain what the MPI_Gather code is doing in this specific example. The code referenced is directly under the "For Question 4.4" comment. 

***Write your answer here***

# Congratulations

You are done with your exam and have completed CMSE 401! I hope you enjoyed your time in this class. Please save the file and upload the jupyter notebook and any other necessary files to the D2L dropbox. 

Written by Dr. Nathan Haut, Michigan State University
<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.

----